IMPORT LIBRARIES

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, classification_report
from sklearn.tree import DecisionTreeClassifier
from scipy import stats

import warnings
warnings.filterwarnings('ignore')

LOAD DATA

In [13]:
pop = pd.read_excel('nt-government-regions_1986-to-2025.xlsx')
crime_nov = pd.read_csv('nt_crime_statistics_nov_2023.csv')
crime_dec = pd.read_csv('nt_crime_statistics_dec_2025.csv')

STANDARDISE COLUMN NAMES

In [14]:
for df in [crime_nov, crime_dec, pop]:
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

CLEAN & SPLIT DATA

In [15]:
crime_nov_hist = (crime_nov[crime_nov['year'] <= 2022]
                  .copy()
                  .rename(columns={'reporting_region': 'region'}))

crime_dec_new  = (crime_dec[crime_dec['reporting_region'] != 'Unknown']
                  .copy()
                  .rename(columns={'reporting_region': 'region',
                                   'offence_type_': 'offence_type'}))

REGION HARMONISATION

In [16]:
region_map = {
    'Alice Springs':  'Central Australia',
    'Darwin':         'Greater Darwin',
    'Palmerston':     'Greater Darwin',
    'Katherine':      'Big Rivers',
    'Tennant Creek':  'Barkly',
    'Nhulunbuy':      'East Arnhem',
    'NT Balance':     'Top End'
}

for df in [crime_nov_hist, crime_dec_new]:
    df['pop_region'] = df['region'].map(region_map)

AGGREGATE CRIME DATA

In [17]:
def aggregate_crime(df):
    return (df.groupby(['year', 'month_number', 'pop_region', 'region', 'offence_category'])
             .agg(total_offences=('number_of_offences', 'sum'))
             .reset_index())

crime_all = (pd.concat([aggregate_crime(crime_nov_hist),
                        aggregate_crime(crime_dec_new)],
                       ignore_index=True)
               .drop_duplicates())

REGION-YEAR-MONTH LEVEL

In [18]:
crime_rym = (crime_all
             .groupby(['year', 'month_number', 'pop_region', 'region'])
             .agg(total_offences=('total_offences', 'sum'))
             .reset_index())

ALCOHOL & DV FEATURES

In [19]:
def get_involvement_flags(df):
    df2 = df.copy()
    df2['alc'] = (df2['alcohol_involvement'].str.lower()
                  .str.contains('yes').fillna(False).astype(int))
    df2['dv']  = (df2['dv_involvement'].str.lower()
                  .str.contains('yes').fillna(False).astype(int))
    return (df2.groupby(['year', 'month_number', 'pop_region', 'region'])
               .agg(alc_offences=('alc', 'sum'),
                    dv_offences=('dv', 'sum'))
               .reset_index())

flags_all = (pd.concat([get_involvement_flags(crime_nov_hist),
                        get_involvement_flags(crime_dec_new)])
               .drop_duplicates())

crime_rym = crime_rym.merge(flags_all,
                             on=['year', 'month_number', 'pop_region', 'region'],
                             how='left')

POPULATION DATA

In [20]:
pop_total = (pop.groupby(['year', 'region'])['population']
               .sum()
               .reset_index()
               .rename(columns={'region': 'pop_region'}))

FEATURE ENGINEERING

In [21]:
merged = (crime_rym
          .merge(pop_total, on=['year', 'pop_region'], how='left')
          .dropna(subset=['population']))

merged['offence_rate_per_1000'] = (merged['total_offences'] / merged['population']) * 1000
merged['alc_rate'] = merged['alc_offences'] / merged['total_offences'].replace(0, np.nan)
merged['dv_rate']  = merged['dv_offences']  / merged['total_offences'].replace(0, np.nan)

merged['log_population'] = np.log(merged['population'])
merged['year_index'] = merged['year'] - merged['year'].min()

merged = merged.fillna(0)

DEMOGRAPHICS

In [22]:
pop_demo = (pop.groupby(['year', 'region'])
               .apply(lambda g: pd.Series({
                   'total_pop': g['population'].sum(),
                   'aboriginal_pop': g.loc[
                       g['aboriginal_status'].str.contains('Aboriginal', na=False),
                       'population'].sum()
               }))
               .reset_index()
               .rename(columns={'region': 'pop_region'}))

pop_demo['aboriginal_pct'] = (pop_demo['aboriginal_pop'] /
                               pop_demo['total_pop'] * 100)

merged2 = (merged
           .merge(pop_demo[['year', 'pop_region', 'aboriginal_pct']],
                  on=['year', 'pop_region'], how='left')
           .fillna(0))

CHECK DATA

In [23]:
print(merged2.head())
print(merged2.shape)

   year  month_number         pop_region         region  total_offences  \
0  2008             1             Barkly  Tennant Creek             137   
1  2008             1         Big Rivers      Katherine             169   
2  2008             1  Central Australia  Alice Springs             420   
3  2008             1        East Arnhem      Nhulunbuy              39   
4  2008             1     Greater Darwin         Darwin             970   

   alc_offences  dv_offences  population  offence_rate_per_1000  alc_rate  \
0             2            2        7322              18.710735  0.014599   
1             2            2       20455               8.262039  0.011834   
2             2            2       40012              10.496851  0.004762   
3             2            2       15322               2.545360  0.051282   
4             2            2      121210               8.002640  0.002062   

    dv_rate  log_population  year_index  aboriginal_pct  
0  0.014599        8.898639 